In [1]:
import torch
import gymnasium as gym
import random
from collections import deque
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import queue

In [6]:
class Qnet(nn.Module):
    def __init__(self, grid_size=10, channels = 4, actions=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size = 3, padding=1), nn.ReLU(),
            nn.Conv2d(32,64,kernel_size = 3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,64,kernel_size = 3,padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * grid_size//2 * grid_size//2, 256), nn.ReLU(),
            nn.Linear(256,actions),
        )
    def forward(self,s):
        
        return self.net(s)
class Buffer:
    def __init__(self, cap = 100000):
        self.buffer = deque(maxlen=cap)
    def add(self, state,action,reward,nextstate,done):
        self.buffer.append((state,action,reward,nextstate,done))
    def sample(self,bsize):
        batch = random.sample(self.buffer,bsize)
        states,actions,rewards,nextstates,dones = zip(*batch)
        return (
            torch.tensor(np.array(states),dtype=torch.float32),
            torch.tensor(actions,dtype=torch.long),
            torch.tensor(rewards,dtype=torch.float32),
            torch.tensor(np.array(nextstates),dtype=torch.float32),
            torch.tensor(dones,dtype=torch.float32),
        )
    def __len__(self):
        return len(self.buffer)
    
        

In [7]:

class SnakeEnv(gym.Env):
    DIRS = [(0, -1), (1, 0), (0, 1), (-1, 0)]

    def __init__(self, grid_size=10, max_steps_without_apple=100):
        super().__init__()
        self.grid_size = grid_size
        self.max_steps_without_apple = max_steps_without_apple
        
        self.action_space = gym.spaces.Discrete(4)
        self.observation_space = gym.spaces.Box(
            low=-1.0, high=20.0, shape=(4,grid_size,grid_size), dtype=np.float32
        )
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mid = self.grid_size // 2
        self.snake = [(mid, mid)]          # head is snake[0]
        self.direction = 1                 # start facing right
        self.apple = self._spawn_apple()
        self.steps = 0
        self.steps_since_apple = 0
        return self._obs(), {}

    def step(self, action):
        if len(self.snake) > 1 and (action + 2) % 4 == self.direction:
            action = self.direction
    
        self.direction = action
        dx, dy = self.DIRS[action]
        head_x, head_y = self.snake[0]
        new_head = (head_x + dx, head_y + dy)
    
        if not (0 <= new_head[0] < self.grid_size and 0 <= new_head[1] < self.grid_size):
            return self._obs(), -1.0, True, False, {}
    
        ate_apple = (new_head == self.apple)
        new_body = [new_head] + (self.snake if ate_apple else self.snake[:-1])
    
        if new_head in new_body[1:]:
            return self._obs(), -1.0, True, False, {}
    
        self.snake = new_body
    
        if ate_apple:
            reward = 1.0
            self.apple = self._spawn_apple()
            self.steps_since_apple = 0
        else:
            old_dist = abs(self.apple[0] - head_x) + abs(self.apple[1] - head_y)
            new_dist = abs(self.apple[0] - new_head[0]) + abs(self.apple[1] - new_head[1])
            
            reward = -0.01 + 0.1 * (old_dist - new_dist)
            self.steps_since_apple += 1
    
        self.steps += 1
        truncated = self.steps_since_apple > self.max_steps_without_apple
        return self._obs(), reward, False, truncated, {}

    def _spawn_apple(self):
        while True:
            pos = (
                self.np_random.integers(0, self.grid_size),
                self.np_random.integers(0, self.grid_size),
            )
            if pos not in self.snake:
                return pos

    def _is_collision(self, pos):
        x, y = pos
        if not (0 <= x < self.grid_size and 0 <= y < self.grid_size):
            return True
        if pos in self.snake[:-1]: 
            return True
        return False
    def _obs(self):
        head_x, head_y = self.snake[0]

        lol = np.zeros((4,10,10),dtype=np.float32)


        lol[0,head_x,head_y] = 1

        for (x,y) in self.snake[1:]: 
            lol[1,x,y] = 1

        lol[2,self.apple[0],self.apple[1]] = 1
        lol[3,0,:]=1
        lol[3,:,0]=1
        lol[3,self.grid_size - 1,:] = 1
        lol[3,:,self.grid_size - 1] = 1
        return lol

    def render(self):
      
        board = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        for i, (x, y) in enumerate(self.snake):
            board[y][x] = 'H' if i == 0 else 'o'
        ax, ay = self.apple
        board[ay][ax] = '*'
        print('\n'.join(''.join(row) for row in board))
        print()

In [8]:
env = SnakeEnv(grid_size=10)

In [9]:
Q = Qnet(grid_size = 10, channels = 4,actions=4)
Qtarget = Qnet(grid_size = 10, channels = 4 ,actions=4)
Qtarget.load_state_dict(Q.state_dict())
buffer = Buffer(cap=100000)

In [10]:
opt = torch.optim.Adam(Q.parameters(),lr=5e-4)
g = 0.99
eps = 1.0
ints = 1000
b = 32
pl = 1000
n = 100000
obs,_ = env.reset()

print(type(obs))
tau = 0.005
episode_return = 0
episode_count = 0
returns = []
best_avg = 0
for step in range(n):
    if random.random() < eps:
        action = env.action_space.sample()
    else:
        with torch.no_grad():
        
            obs_t = torch.from_numpy(obs).unsqueeze(0)
            action = Q(obs_t).argmax().item()

    nextobs, r, term, trun, _ = env.step(action)
    episode_return += r
    done = term or trun
   
    buffer.add(obs,action,r,nextobs,done)
    obs = nextobs if not done else env.reset()[0]
    if done:
        returns.append(episode_return)
        episode_count += 1
        if episode_count % 10 == 0:
            avg = sum(returns[-10:]) / 10
            if avg > best_avg:
                best_avg = avg
                torch.save(Q.state_dict(),"best.pt")
            print(f"episode {episode_count}, avg return (last 10): {avg:.1f}, epsilon: {eps:.3f}")
        episode_return = 0
    if len(buffer) > ints:
        states, actions,rs,nexts,dones = buffer.sample(b)
       
        q_values = Q(states)
        predicted = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_actions = Q(nexts).argmax(dim=1,keepdim=True)
            next_q = Qtarget(nexts).gather(1,next_actions).squeeze(1)
            target = rs + g * next_q * (1 - dones)
        loss = F.smooth_l1_loss(predicted,target)
        #if step % 100 == 0 : print(step,loss.item())
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(Q.parameters(), 10.0)   
        opt.step()
    with torch.no_grad():
        for tp, op in zip(Qtarget.parameters(), Q.parameters()):
            tp.data.mul_(1 - tau).add_(op.data, alpha=tau)
    eps = max(0.05, eps * 0.9995)

<class 'numpy.ndarray'>
episode 10, avg return (last 10): -1.4, epsilon: 0.861
episode 20, avg return (last 10): -1.2, epsilon: 0.789
episode 30, avg return (last 10): -1.3, epsilon: 0.726
episode 40, avg return (last 10): -1.3, epsilon: 0.674
episode 50, avg return (last 10): -1.1, epsilon: 0.639
episode 60, avg return (last 10): -1.0, epsilon: 0.603
episode 70, avg return (last 10): -1.1, epsilon: 0.469
episode 80, avg return (last 10): 0.2, epsilon: 0.390
episode 90, avg return (last 10): 0.7, epsilon: 0.350
episode 100, avg return (last 10): 0.7, epsilon: 0.308
episode 110, avg return (last 10): 0.7, epsilon: 0.280
episode 120, avg return (last 10): 1.8, epsilon: 0.219
episode 130, avg return (last 10): 1.6, epsilon: 0.185
episode 140, avg return (last 10): 2.7, epsilon: 0.140
episode 150, avg return (last 10): 2.6, epsilon: 0.101
episode 160, avg return (last 10): 3.5, epsilon: 0.076
episode 170, avg return (last 10): 9.1, epsilon: 0.050
episode 180, avg return (last 10): 9.6, eps